In [1]:
!pip install librosa soundfile scikit-learn matplotlib

In [2]:
import numpy as np
import librosa
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Libraries imported successfully!")

Libraries imported successfully!


In [6]:
from google.colab import files

uploaded = files.upload()

Saving archive.zip to archive.zip


In [7]:

import zipfile
import os

zip_file = next(
    (f for f in uploaded if f.lower().endswith(".zip")),
    None
)

if zip_file:
    with zipfile.ZipFile(zip_file, "r") as z:
        z.extractall("/content/ravdess")

    print("Dataset extracted successfully!")
    print("Files and folders:", os.listdir("/content/ravdess")[:10])
else:
    print("ZIP file not found. Please upload the dataset ZIP.")


Dataset extracted successfully!
Files and folders: ['Actor_02', 'Actor_04', 'Actor_06', 'Actor_07', 'Actor_10', 'Actor_22', 'Actor_21', 'Actor_19', 'Actor_05', 'Actor_03']


In [8]:

import os
import numpy as np
import librosa

audio_data = []
labels = []

# RAVDESS emotion codes
emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

for root, dirs, files_list in os.walk("/content/ravdess"):
    for file in files_list:
        if file.endswith(".wav"):
            parts = file.split("-")

            if len(parts) >= 3 and parts[2] in emotion_map:
                emotion = emotion_map[parts[2]]
                path = os.path.join(root, file)

                try:
                    audio, sr = librosa.load(path, duration=3, sr=22050)
                    mfcc = librosa.feature.mfcc(
                        y=audio, sr=sr, n_mfcc=40
                    )
                    features = np.mean(mfcc.T, axis=0)

                    audio_data.append(features)
                    labels.append(emotion)

                except Exception as e:
                    print("Skipped:", file, e)

print("Audio files processed:", len(audio_data))
print("Emotions found:", sorted(set(labels)))


Audio files processed: 2880
Emotions found: ['angry', 'calm', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised']


In [9]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X = np.array(audio_data)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100, random_state=42
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Model training completed!")
print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("\nClassification Report:")
print(classification_report(y_test, predictions))


Model training completed!
Accuracy: 0.931

Classification Report:
              precision    recall  f1-score   support

       angry       0.95      1.00      0.97        76
        calm       0.88      0.92      0.90        77
     disgust       0.92      0.95      0.94        77
     fearful       0.95      0.95      0.95        77
       happy       0.92      0.95      0.94        77
     neutral       1.00      0.89      0.94        38
         sad       0.89      0.84      0.87        77
   surprised       0.97      0.92      0.95        77

    accuracy                           0.93       576
   macro avg       0.94      0.93      0.93       576
weighted avg       0.93      0.93      0.93       576



In [10]:

import joblib

joblib.dump(model, "emotion_recognition_model.pkl")

print("Model saved successfully!")


Model saved successfully!
